# LLM Provider 完整指南

ADLab 支持多种 LLM Provider，包括 OpenAI、Claude、vLLM 和 SGLang。

## 目录
1. [LanguageModel 基类](#1-LanguageModel-基类)
2. [OpenAI API](#2-OpenAI-API)
3. [Claude API](#3-Claude-API)
4. [vLLM 和 SGLang](#4-vLLM-和-SGLang)
5. [使用示例](#5-使用示例)

## 1. LanguageModel 基类

In [ ]:
from algodisco.base.llm import LanguageModel
import inspect

# 查看 LanguageModel 的方法
print("LanguageModel 方法:")
for name, method in inspect.getmembers(LanguageModel, predicate=inspect.isfunction):
    if not name.startswith('_'):
        print(f"  - {name}")

## 2. OpenAI API

In [ ]:
import os

# 设置 API Key (请替换为你的 key)
os.environ["OPENAI_API_KEY"] = "your-api-key"
os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"

from algodisco.providers.llm import OpenAIAPI

# 初始化
llm = OpenAIAPI(
    model="gpt-3.5-turbo",
    # base_url 和 api_key 可以不传，会从环境变量读取
)

## 3. Claude API

In [ ]:
from algodisco.providers.llm import ClaudeAPI

# 初始化 Claude
claude = ClaudeAPI(
    model="claude-3-sonnet-20240229",
    api_key="your-anthropic-api-key"
)

# 注意: Claude 也支持通过环境变量 ANTHROPIC_API_KEY 设置

## 4. vLLM 和 SGLang

In [ ]:
# vLLM 和 SGLang 是本地部署的 LLM 服务器

# vLLM 服务器启动:
# vllm serve Q/Qwen2.5-0.5B-Instruct --dtype half --api-key token-abc123

# SGLang 服务器启动:
# python -m sglang.launch_server --model Q/Qwen2.5-0.5B-Instruct --dtype half --api-key token-abc123

from algodisco.providers.llm import VLLMServer, SGLangServer

# vLLM 初始化
vllm = VLLMServer(
    model="Q/Qwen2.5-0.5B-Instruct",
    base_url="http://localhost:8000/v1",
    api_key="token-abc123"
)

# SGLang 初始化
sglang = SGLangServer(
    model="Q/Qwen2.5-0.5B-Instruct",
    base_url="http://localhost:30000/v1",
    api_key="token-abc123"
)

## 5. 使用示例

In [ ]:
# 统一接口 - chat_completion

# 基本用法 - 字符串消息
response = llm.chat_completion(
    message="Hello, how are you?",
    max_tokens=100,
    timeout_seconds=30
)

print(f"回复: {response[:100]}...")

In [ ]:
# 高级用法 - 消息列表（多轮对话）

messages = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Write a Python function to sort a list."},
]

response = llm.chat_completion(
    message=messages,
    max_tokens=200,
    timeout_seconds=30
)

print(f"回复: {response[:200]}...")

In [ ]:
# 使用额外参数

response = llm.chat_completion(
    message="Write a haiku about programming",
    max_tokens=50,
    temperature=0.7,  # 控制随机性
    top_p=0.9,      # 核采样
)

print(f"回复: {response}")

In [ ]:
# Embedding 用法

# 单个文本
embedding = llm.embedding(
    text="Hello world",
    dimensions=1536
)
print(f"Embedding 维度: {len(embedding)}")

# 多个文本
embeddings = llm.embedding(
    text=["Hello", "World", "Test"]
)
print(f"Embedding 数量: {len(embeddings)}")

In [ ]:
# 资源管理 - close()

llm.close()
claude.close()
vllm.close()
sglang.close()

# 或者使用上下文管理器（如果实现了 __enter__/__exit__）
# with OpenAIAPI(model="gpt-3.5-turbo") as llm:
#     response = llm.chat_completion("Hello")

## YAML 配置示例

In [ ]:
# OpenAI 配置
openai_config = """
llm:
  class_path: "algodisco.providers.llm.openai_api.OpenAIAPI"
  kwargs:
    model: "gpt-3.5-turbo"
    api_key: "your-api-key"  # 或通过环境变量
    base_url: "https://api.openai.com/v1"
"""

# Claude 配置
claude_config = """
llm:
  class_path: "algodisco.providers.llm.claude_api.ClaudeAPI"
  kwargs:
    model: "claude-3-sonnet-20240229"
    api_key: "your-anthropic-api-key"
"""

# vLLM 配置
vllm_config = """
llm:
  class_path: "algodisco.providers.llm.vllm_server.VLLMServer"
  kwargs:
    model: "Q/Qwen2.5-0.5B-Instruct"
    base_url: "http://localhost:8000/v1"
    api_key: "token-abc123"
"""

print("YAML 配置示例:")
print(openai_config)

## 总结

| Provider | 类 | 用途 |
|----------|------|------|
| OpenAI | `OpenAIAPI` | OpenAI API 兼容 |
| Claude | `ClaudeAPI` | Anthropic Claude |
| vLLM | `VLLMServer` | 本地 vLLM 服务器 |
| SGLang | `SGLangServer` | 本地 SGLang 服务器 |

| 方法 | 说明 |
|--------|------|
| `chat_completion()` | 发送聊天请求 |
| `embedding()` | 生成嵌入向量 |
| `close()` | 释放资源 |